# PropAI — AI Investment Memo Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/propai/blob/main/notebooks/03_ai_memo_generation.ipynb)

This notebook shows the full PropAI AI pipeline:

1. **Parse** a plain-English deal description → structured `DealInput` (Claude Sonnet)
2. **Underwrite** — run the financial engine (pure Python, no AI)
3. **Generate** a 9-section institutional investment memo (Claude Opus)

**Requires:** `ANTHROPIC_API_KEY` set as an environment variable or Colab secret.

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'anthropic', 'httpx', 'jinja2', 'weasyprint', '-q'],
    check=True
)

import os
if not os.path.exists('propai'):
    os.system('git clone https://github.com/your-org/propai.git --depth=1 -q')
sys.path.insert(0, os.path.join(os.getcwd(), 'propai', 'backend'))
print('Setup complete')

In [ ]:
import os

# Set your Anthropic API key
# In Colab: use Secrets (key icon in sidebar) → name it ANTHROPIC_API_KEY
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab Secrets')
except Exception:
    if not os.environ.get('ANTHROPIC_API_KEY'):
        print('⚠️  Set ANTHROPIC_API_KEY env var to run AI sections')
        print('   Sections 1–3 (underwriting) work without a key.')
    else:
        print('API key found in environment')

## 1. Parse a Deal from Plain English

In [ ]:
import asyncio
from agents.deal_parser import DealParser

deal_description = """
24-unit apartment complex in Austin, TX. Purchase price $4.8M.
Average rents are $2,000/month. 70% LTV at 6.75% on a 30-year amortization.
Planning a 5-year hold with an exit at a 5.5 cap rate.
Property taxes $72k/year, insurance $18k, management 5% of EGI.
"""

async def parse():
    parser = DealParser()
    return await parser.parse(deal_description)

parse_result = asyncio.run(parse())
deal = parse_result.deal_input

print('Parsed Deal:')
print(f'  Name:            {deal.name}')
print(f'  Asset Class:     {deal.asset_class}')
print(f'  Purchase Price:  ${deal.purchase_price:,.0f}')
print(f'  Market:          {deal.market}')
print(f'  Units:           {deal.units}')
print(f'  GSI:             ${deal.operations.gross_scheduled_income:,.0f}/yr')
print(f'  LTV:             {deal.loan.ltv:.0%}')
print(f'  Interest Rate:   {deal.loan.interest_rate:.2%}')
print(f'  Hold Period:     {deal.exit.hold_period_years} years')

if parse_result.assumed_values:
    print(f'\nDefault assumptions applied: {list(parse_result.assumed_values.keys())}')
if parse_result.clarifications_needed:
    print(f'\nClarifications that would improve accuracy:')
    for c in parse_result.clarifications_needed:
        print(f'  ? {c}')

## 2. Run the Financial Engine

In [ ]:
from engine.financial.proforma import ProFormaEngine

engine = ProFormaEngine(deal)
result = engine.underwrite()
m = result.metrics

print('Key Return Metrics')
print('=' * 40)
print(f'  Cap Rate:         {m.going_in_cap_rate:.2%}')
print(f'  GRM:              {m.gross_rent_multiplier:.2f}x')
print(f'  DSCR (Yr 1):      {m.dscr_yr1:.2f}x')
print(f'  Cash-on-Cash:     {m.cash_on_cash_yr1:.1%}')
print(f'  Levered IRR:      {m.levered_irr:.1%}')
print(f'  Equity Multiple:  {m.equity_multiple:.2f}x')
print(f'  Levered NPV:      ${m.levered_npv:,.0f}')
print(f'  Equity Required:  ${m.equity_required:,.0f}')
print('=' * 40)

if result.warnings:
    print('\n⚠️  Warnings:')
    for w in result.warnings:
        print(f'   {w}')

## 3. Generate the Investment Memo with Claude

PropAI calls Claude section-by-section for higher quality and better token management. Each section gets full deal context but focused instructions.

In [ ]:
import os

if not os.environ.get('ANTHROPIC_API_KEY'):
    print('No ANTHROPIC_API_KEY found — skipping memo generation.')
    print('Set the key to generate a full memo.')
else:
    from agents.memo_agent import MemoAgent

    async def generate_memo():
        agent = MemoAgent()
        return await agent.generate(deal, result)

    print('Generating investment memo... (15–30 seconds)')
    memo = asyncio.run(generate_memo())
    print(f'\n✅ Memo generated: {memo.deal_name}')

In [ ]:
if 'memo' in dir() and memo:
    print('=== EXECUTIVE SUMMARY ===')
    print(memo.sections.get('executive_summary', '')[:1500])
    print('...')

In [ ]:
if 'memo' in dir() and memo:
    print('=== INVESTMENT HIGHLIGHTS ===')
    print(memo.sections.get('investment_highlights', ''))
    print()
    print('=== RISK FACTORS ===')
    print(memo.sections.get('risk_factors', ''))

## 4. Render to HTML / PDF

In [ ]:
if 'memo' in dir() and memo and memo.html:
    # Save HTML
    html_path = '/tmp/propai_memo.html'
    with open(html_path, 'w') as f:
        f.write(memo.html)
    print(f'HTML memo saved to {html_path}')

    # Display inline
    from IPython.display import HTML
    display(HTML(f'<iframe src="{html_path}" width="100%" height="800px"></iframe>'))

In [ ]:
if 'memo' in dir() and memo and memo.html:
    try:
        from weasyprint import HTML
        pdf_path = '/tmp/propai_memo.pdf'
        HTML(string=memo.html).write_pdf(pdf_path)
        print(f'PDF saved to: {pdf_path}')

        # Download in Colab
        try:
            from google.colab import files
            files.download(pdf_path)
        except Exception:
            print('Open the file at the path above to view the PDF.')
    except ImportError:
        print('Install weasyprint for PDF export: pip install weasyprint')

## 5. Full Pipeline — One Call

The `/api/ai/analyze` endpoint does all three steps (parse → underwrite → memo) in one request.

In [ ]:
# If you have the PropAI server running locally:
# docker-compose up

import httpx

PROPAI_URL = 'http://localhost:8000'

try:
    resp = httpx.get(f'{PROPAI_URL}/health', timeout=2)
    if resp.status_code == 200:
        print('PropAI server is running. Running full pipeline...')

        result_api = httpx.post(
            f'{PROPAI_URL}/api/ai/analyze',
            json={"text": deal_description},
            timeout=120,
        ).json()

        u = result_api['underwriting']
        print(f"Levered IRR:    {u['metrics']['levered_irr']:.1%}")
        print(f"Equity Multiple:{u['metrics']['equity_multiple']:.2f}x")
        print(f"Memo sections:  {list(result_api['memo']['sections'].keys())}")
    else:
        print(f'Server returned {resp.status_code}')
except Exception as e:
    print(f'Server not reachable ({e})')
    print('Run `docker-compose up` to start PropAI locally')

---
## Summary

In this notebook we:
1. Parsed a plain-English deal description into structured data using Claude Sonnet
2. Ran the pure-Python financial engine to compute IRR, DSCR, equity multiple
3. Generated a full institutional investment memo with Claude Opus
4. Exported to HTML and PDF

**The financial numbers came from the engine, not the AI.** Claude only writes prose.
This is intentional — it makes the output auditable and the assumptions transparent.

---

- **[Notebook 1](./01_underwriting_demo.ipynb)** — Deep dive into the financial engine
- **[Notebook 2](./02_market_analysis.ipynb)** — Market data from Census, FRED, HUD, Zillow
- **[GitHub](https://github.com/your-org/propai)** — Star the repo ⭐